In [ ]:
import pandas as pd

# 1. 데이터 불러오기
weather_df = pd.read_csv('/content/[Ina]train_clean_final.csv', parse_dates=['ymdt'])
traff_df = pd.read_csv('/content/지점별_시간대별_평균_지역구포함.csv')

# 2. 타입 정리
weather_df['station_name'] = weather_df['station_name'].astype(str)
traff_df['지역구'] = traff_df['지역구'].astype(str)

# 3. traff_df를 long format으로 변환 (3시, 8시, 14시, 18시만)
traff_long = traff_df.melt(
    id_vars=['일자', '지역구', '지점번호'],
    value_vars=['3시', '8시', '14시', '18시'],
    var_name='시간',
    value_name='교통량'
)

# 4. 시간 정리: '3시' -> 3
traff_long['시간'] = traff_long['시간'].str.replace('시', '').astype(int)

# 5. 일자 → datetime 변환
traff_long['일자'] = pd.to_datetime(traff_long['일자'].astype(str), format='%Y%m%d')

# 6. 시간 더해서 full datetime 생성
traff_long['시간_dt'] = traff_long['시간'].apply(lambda h: pd.Timedelta(hours=h))
traff_long['datetime'] = traff_long['일자'] + traff_long['시간_dt']

# 7. 병합: 날짜+시간과 지역구 기준
merged_df = pd.merge(
    weather_df,
    traff_long[['datetime', '지역구', '교통량']],
    left_on=['ymdt', 'station_name'],
    right_on=['datetime', '지역구'],
    how='left'
)

# 8. 불필요한 컬럼 제거
merged_df = merged_df.drop(columns=['datetime', '지역구'])

# 9. 결과 확인
print(f"병합 후 행 수: {len(merged_df)} (원본 weather_df: {len(weather_df)})")
print(merged_df.head(10))

병합 후 행 수: 20257 (원본 weather_df: 20257)
   id station_name                ymdt  season park1_name  park1_dir_sin  \
0   1          강남구 2024-01-01 03:00:00  winter        서울숲      -0.316228   
1   2          강동구 2024-01-01 03:00:00  winter      올림픽공원      -0.447214   
2   3          강북구 2024-01-01 03:00:00  winter     북서울꿈의숲       0.650791   
3   4          강서구 2024-01-01 03:00:00  winter      선유도공원       0.995893   
4   5          관악구 2024-01-01 03:00:00  winter        현충원       0.371391   
5   6          광진구 2024-01-01 03:00:00  winter      올림픽공원       0.554700   
6   7          구로구 2024-01-01 03:00:00  winter      보라매공원       0.928477   
7   8          금천구 2024-01-01 03:00:00  winter      보라매공원       0.216930   
8   9          노원구 2024-01-01 03:00:00  winter     북서울꿈의숲      -0.406138   
9  11         동대문구 2024-01-01 03:00:00  winter      푸른식물원      -0.800000   

   park1_dir_cos  park1_distance  park1_area park2_name  ...  weekday  \
0      -0.948683        6.324555     1156498     매헌

In [ ]:
# 10. CSV 파일로 저장
merged_df.to_csv('/content/final_traffic_HR.csv', index=False)
